# 面试问题：INT8 PTQ 与 QAT 如何从零实现，校准数据为什么重要？

可以直接复述的回答是：对称 INT8 量化先用 `scale=max(abs(x))/127` 把浮点值映射到 `[-127,127]`，整数计算后再乘 scale 还原。PTQ 在模型训练完成后量化，成本低，但 activation scale 完全依赖校准集。若校准集遗漏高幅值样本，线上激活会饱和到 127。QAT 在训练前向插入 fake quant，反向用 straight-through estimator，让权重适应舍入噪声。应同时报告精度、饱和率、量化误差、模型大小和目标硬件真实延迟。下面手写 scale、round、clamp、dequant 与 STE，不调用量化框架。

## 真实案例：压缩边缘设备电机风险回归模型

十条传感器记录包含振动、温升、电流和风险分数，最后两条是高幅值异常。数据为教学构造的脱敏结构，目的是让“不具代表性的校准集”产生可观察饱和，不可外推真实设备故障率。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(11)  # 固定模型初始化和训练结果
records = [  # 定义十条带业务语义的电机传感器记录
    ("M-01", 0.20, 0.10, 0.30),  # 健康低幅值记录
    ("M-02", 0.40, 0.30, 0.50),  # 健康轻微波动记录
    ("M-03", 0.70, 0.40, 0.60),  # 普通运行记录
    ("M-04", 0.30, 0.80, 0.70),  # 温升稍高记录
    ("M-05", 0.90, 0.60, 1.00),  # 中等振动记录
    ("M-06", 1.20, 0.90, 1.10),  # 较高负载记录
    ("M-07", 1.50, 1.20, 1.40),  # 告警边界记录
    ("M-08", 1.80, 1.50, 1.60),  # 校准集最高普通记录
    ("M-09", 4.50, 3.80, 5.00),  # 高幅值轴承异常记录
    ("M-10", 5.50, 4.50, 6.00),  # 极端过载记录
]  # 结束十条传感器记录
x = torch.tensor([[row[1], row[2], row[3]] for row in records], dtype=torch.float32)  # 构造三维传感器输入
true_weight = torch.tensor([0.45, 0.25, 0.30])  # 定义教学风险生成规则权重
y = torch.relu(x @ true_weight - 0.10).unsqueeze(1)  # 生成连续风险监督目标
print("输入预览：id | 振动 | 温升 | 电流 | 风险目标")  # 输出传感器字段标题
for index, row in enumerate(records):  # 逐条展示十个设备样本
    print(f"{row[0]} | {row[1]:4.2f} | {row[2]:4.2f} | {row[3]:4.2f} | {float(y[index]):5.3f}")  # 展示原始输入和目标风险
print("普通校准候选=前8条，异常部署输入=后2条")  # 明确失败场景的数据切分

输入预览：id | 振动 | 温升 | 电流 | 风险目标
M-01 | 0.20 | 0.10 | 0.30 | 0.105
M-02 | 0.40 | 0.30 | 0.50 | 0.305
M-03 | 0.70 | 0.40 | 0.60 | 0.495
M-04 | 0.30 | 0.80 | 0.70 | 0.445
M-05 | 0.90 | 0.60 | 1.00 | 0.755
M-06 | 1.20 | 0.90 | 1.10 | 0.995
M-07 | 1.50 | 1.20 | 1.40 | 1.295
M-08 | 1.80 | 1.50 | 1.60 | 1.565
M-09 | 4.50 | 3.80 | 5.00 | 4.375
M-10 | 5.50 | 4.50 | 6.00 | 5.300
普通校准候选=前8条，异常部署输入=后2条


## Baseline / 基线：先训练 FP32 小网络

模型包含一个四单元 ReLU 隐层。训练使用真实 forward/backward 和手写 SGD，随后 PTQ 才能与同一 FP32 输出比较。

In [2]:
class RiskNet(torch.nn.Module):  # 定义从零参数化的两层风险网络
    def __init__(self):  # 初始化两层权重和偏置
        super().__init__()  # 初始化 PyTorch 模块基类
        self.weight1 = torch.nn.Parameter(torch.randn(3, 4) * 0.20)  # 创建输入到隐层权重
        self.bias1 = torch.nn.Parameter(torch.zeros(4))  # 创建隐层偏置
        self.weight2 = torch.nn.Parameter(torch.randn(4, 1) * 0.20)  # 创建隐层到输出权重
        self.bias2 = torch.nn.Parameter(torch.zeros(1))  # 创建输出偏置
    def forward(self, features):  # 定义 FP32 风险预测前向传播
        hidden = torch.relu(features @ self.weight1 + self.bias1)  # 计算四维 ReLU 隐层激活
        return hidden @ self.weight2 + self.bias2  # 计算连续风险分数
fp_model = RiskNet()  # 创建待训练 FP32 网络
fp_trace = []  # 保存关键训练步损失和梯度
for step in range(1, 501):  # 对十条记录执行五百步全批量训练
    prediction = fp_model(x)  # 真实执行网络 forward
    loss = ((prediction - y) ** 2).mean()  # 计算风险均方误差
    loss.backward()  # 真实执行 backward 得到两层参数梯度
    gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in fp_model.parameters()))  # 计算全局梯度范数
    with torch.no_grad():  # 关闭手写 SGD 更新的梯度记录
        for parameter in fp_model.parameters():  # 遍历网络四个参数张量
            parameter -= 0.01 * parameter.grad  # 使用固定学习率更新参数
            parameter.grad.zero_()  # 清空本步梯度
    if step in {1, 20, 100, 500}:  # 保留关键收敛节点
        fp_trace.append((step, float(loss), float(gradient_norm)))  # 保存真实训练轨迹
with torch.no_grad():  # 进入 FP32 评估阶段
    fp_prediction = fp_model(x)  # 计算十条记录 FP32 预测
fp_mse = float(((fp_prediction - y) ** 2).mean())  # 计算 FP32 基线均方误差
print("FP32 训练：step | MSE | grad_norm")  # 输出训练轨迹表头
for item in fp_trace:  # 遍历关键训练节点
    print(f"{item[0]:4d} | {item[1]:.6f} | {item[2]:.6f}")  # 展示损失和梯度真实变化
print(f"FP32 最终 MSE={fp_mse:.6f}")  # 输出后续量化比较基线

FP32 训练：step | MSE | grad_norm
   1 | 5.684798 | 6.505903
  20 | 0.122157 | 1.070036
 100 | 0.011570 | 0.169385
 500 | 0.001722 | 0.009223
FP32 最终 MSE=0.001721


## 核心实现：手写对称 INT8 PTQ 与校准 scale

权重 scale 来自完整权重张量；activation scale 来自离线校准数据。基线只校准前八条普通记录，修正方案把两条异常范围也纳入代表性校准。

In [3]:
def symmetric_scale(tensor):  # 按张量绝对最大值计算对称 INT8 scale
    maximum = float(tensor.detach().abs().max())  # 提取不参与梯度的最大幅值
    return maximum / 127.0 if maximum > 0.0 else 1.0  # 避免全零张量产生零 scale
def quantize_dequantize(tensor, scale):  # 手写浮点到 INT8 再反量化过程
    integer = torch.round(tensor / scale).clamp(-127, 127).to(torch.int8)  # 执行缩放、舍入和饱和截断
    restored = integer.float() * scale  # 把整数码还原为近似浮点值
    return integer, restored  # 返回可观察整数码和反量化张量
with torch.no_grad():  # 使用训练后模型采集激活范围
    bad_calibration_hidden = torch.relu(x[:8] @ fp_model.weight1 + fp_model.bias1)  # 只用普通记录采集隐层范围
    good_calibration_hidden = torch.relu(x @ fp_model.weight1 + fp_model.bias1)  # 用覆盖异常范围的代表性记录采集隐层
bad_input_scale = symmetric_scale(x[:8])  # 计算不完整输入校准 scale
good_input_scale = symmetric_scale(x)  # 计算代表性输入校准 scale
bad_hidden_scale = symmetric_scale(bad_calibration_hidden)  # 计算不完整隐层校准 scale
good_hidden_scale = symmetric_scale(good_calibration_hidden)  # 计算代表性隐层校准 scale
def ptq_forward(model, features, input_scale, hidden_scale):  # 手写两层网络 PTQ 模拟前向
    input_int, restored_input = quantize_dequantize(features, input_scale)  # 量化并还原输入 activation
    weight1_int, restored_weight1 = quantize_dequantize(model.weight1, symmetric_scale(model.weight1))  # 量化并还原第一层权重
    hidden = torch.relu(restored_input @ restored_weight1 + model.bias1)  # 用量化输入和权重计算隐层
    hidden_int, restored_hidden = quantize_dequantize(hidden, hidden_scale)  # 量化并还原隐层 activation
    weight2_int, restored_weight2 = quantize_dequantize(model.weight2, symmetric_scale(model.weight2))  # 量化并还原第二层权重
    output = restored_hidden @ restored_weight2 + model.bias2  # 计算反量化风险输出
    return output, {"input_int": input_int, "weight1_int": weight1_int, "hidden_int": hidden_int, "weight2_int": weight2_int}  # 返回预测和整数中间量
with torch.no_grad():  # 禁止 PTQ 评估建立梯度图
    bad_ptq_prediction, bad_codes = ptq_forward(fp_model, x, bad_input_scale, bad_hidden_scale)  # 执行不具代表性校准 PTQ
    good_ptq_prediction, good_codes = ptq_forward(fp_model, x, good_input_scale, good_hidden_scale)  # 执行代表性校准 PTQ
bad_ptq_mse = float(((bad_ptq_prediction - y) ** 2).mean())  # 计算错误校准 PTQ 误差
good_ptq_mse = float(((good_ptq_prediction - y) ** 2).mean())  # 计算修正校准 PTQ 误差
bad_saturation = int((bad_codes["input_int"].abs() == 127).sum())  # 统计错误校准下输入饱和码数量
good_saturation = int((good_codes["input_int"].abs() == 127).sum())  # 统计代表性校准下输入饱和码数量
print(f"输入 scale：普通校准={bad_input_scale:.6f}，代表性校准={good_input_scale:.6f}")  # 展示校准范围如何改变 scale
print(f"隐层 scale：普通校准={bad_hidden_scale:.6f}，代表性校准={good_hidden_scale:.6f}")  # 展示隐层激活范围差异
print(f"PTQ MSE：普通校准={bad_ptq_mse:.6f}，代表性校准={good_ptq_mse:.6f}")  # 对比同模型量化误差
print(f"输入饱和码数：普通校准={bad_saturation}，代表性校准={good_saturation}")  # 量化失败的直接证据
print("第一层 INT8 权重码：", good_codes["weight1_int"].T.tolist())  # 展示真实整数权重而非只打印断言

输入 scale：普通校准=0.014173，代表性校准=0.047244
隐层 scale：普通校准=0.014695，代表性校准=0.052788
PTQ MSE：普通校准=2.275555，代表性校准=0.002081
输入饱和码数：普通校准=7，代表性校准=1
第一层 INT8 权重码： [[43, -9, 31], [62, -21, -50], [47, 63, 127], [-50, 18, -12]]


## QAT：用 STE 让权重适应舍入误差

`round` 的真实导数几乎处处为零。`x + (qdq(x)-x).detach()` 让前向看到量化值、反向把梯度近似直传给浮点 master weight。

In [4]:
def fake_quantize_ste(tensor, scale):  # 手写带直通估计器的 fake quant
    integer = torch.round(tensor / scale).clamp(-127, 127)  # 前向计算浮点形式的 INT8 码
    restored = integer * scale  # 计算反量化前向值
    return tensor + (restored - tensor).detach()  # 前向使用 restored 而反向对 tensor 传单位梯度
class QATRiskNet(RiskNet):  # 定义插入 fake quant 的量化感知网络
    def forward(self, features):  # 定义 QAT 前向传播
        quantized_input = fake_quantize_ste(features, good_input_scale)  # 用固定代表性 scale 量化输入
        weight1_scale = symmetric_scale(self.weight1)  # 从当前第一层浮点权重计算 scale
        quantized_weight1 = fake_quantize_ste(self.weight1, weight1_scale)  # 对第一层权重执行 STE fake quant
        hidden = torch.relu(quantized_input @ quantized_weight1 + self.bias1)  # 计算带量化噪声隐层
        quantized_hidden = fake_quantize_ste(hidden, good_hidden_scale)  # 对隐层 activation 执行 fake quant
        weight2_scale = symmetric_scale(self.weight2)  # 从当前第二层权重计算 scale
        quantized_weight2 = fake_quantize_ste(self.weight2, weight2_scale)  # 对第二层权重执行 STE fake quant
        return quantized_hidden @ quantized_weight2 + self.bias2  # 计算带量化噪声风险输出
qat_model = QATRiskNet()  # 创建量化感知训练模型
qat_model.load_state_dict(fp_model.state_dict())  # 从训练好的 FP32 权重开始微调
qat_trace = []  # 保存 QAT 损失和梯度轨迹
for step in range(1, 121):  # 执行一百二十步短量化感知微调
    qat_prediction_train = qat_model(x)  # 前向插入输入、权重和隐层 fake quant
    qat_loss = ((qat_prediction_train - y) ** 2).mean()  # 计算量化路径风险均方误差
    qat_loss.backward()  # 通过 STE 真实执行 backward
    qat_gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in qat_model.parameters()))  # 计算 QAT 全局梯度范数
    with torch.no_grad():  # 关闭手写更新的计算图
        for parameter in qat_model.parameters():  # 遍历 QAT 浮点 master 参数
            parameter -= 0.003 * parameter.grad  # 用小学习率适应舍入噪声
            parameter.grad.zero_()  # 清空本步梯度
    if step in {1, 10, 40, 120}:  # 保留关键 QAT 节点
        qat_trace.append((step, float(qat_loss), float(qat_gradient_norm)))  # 保存损失与梯度轨迹
with torch.no_grad():  # 进入 QAT 模型评估阶段
    qat_prediction = qat_model(x)  # 获取 fake quant 路径最终预测
qat_mse = float(((qat_prediction - y) ** 2).mean())  # 计算量化感知模型误差
print("QAT：step | quantized MSE | grad_norm")  # 输出 QAT 真实训练轨迹表头
for item in qat_trace:  # 遍历四个训练节点
    print(f"{item[0]:4d} | {item[1]:.6f} | {item[2]:.6f}")  # 展示 STE 梯度和量化损失下降
print(f"代表性 PTQ MSE={good_ptq_mse:.6f}，QAT MSE={qat_mse:.6f}")  # 对比训练后量化路径误差

QAT：step | quantized MSE | grad_norm
   1 | 0.002081 | 0.014660
  10 | 0.002080 | 0.012702
  40 | 0.001521 | 0.017373
 120 | 0.001975 | 0.010592
代表性 PTQ MSE=0.002081，QAT MSE=0.001975


## 失败案例与修正、逐样本结果

前八条普通记录给出的 scale 太小，M-09/M-10 多个输入都被夹到 127，风险被严重低估。纳入异常范围后饱和显著减少；QAT 再让权重适应离散码点。

In [5]:
fp_bytes = sum(parameter.numel() * 4 for parameter in fp_model.parameters())  # 按 FP32 四字节估算原模型参数体积
int8_bytes = sum(parameter.numel() for name, parameter in fp_model.named_parameters() if "weight" in name) + sum(parameter.numel() * 4 for name, parameter in fp_model.named_parameters() if "bias" in name)  # 按权重 INT8 偏置 FP32 估算量化体积
print("id | target | FP32 | 错误 PTQ | 修正 PTQ | QAT | 错误输入最大码")  # 输出逐样本量化对照表头
for index, row in enumerate(records):  # 遍历十条设备记录
    maximum_code = int(bad_codes["input_int"][index].abs().max())  # 获取错误校准下当前样本最大整数码
    print(f"{row[0]} | {float(y[index]):6.3f} | {float(fp_prediction[index]):6.3f} | {float(bad_ptq_prediction[index]):8.3f} | {float(good_ptq_prediction[index]):8.3f} | {float(qat_prediction[index]):6.3f} | {maximum_code:3d}")  # 展示饱和对高风险预测的影响
print(f"参数存储估算：FP32={fp_bytes} bytes，INT8权重+FP32偏置={int8_bytes} bytes，压缩={fp_bytes / int8_bytes:.2f}x")  # 展示体积收益与偏置例外

id | target | FP32 | 错误 PTQ | 修正 PTQ | QAT | 错误输入最大码
M-01 |  0.105 |  0.133 |    0.127 |    0.110 |  0.109 |  21
M-02 |  0.305 |  0.328 |    0.321 |    0.313 |  0.312 |  35
M-03 |  0.495 |  0.469 |    0.466 |    0.489 |  0.488 |  49
M-04 |  0.445 |  0.524 |    0.521 |    0.543 |  0.542 |  56
M-05 |  0.755 |  0.770 |    0.779 |    0.769 |  0.779 |  71
M-06 |  0.995 |  0.960 |    0.965 |    0.945 |  0.943 |  85
M-07 |  1.295 |  1.253 |    1.251 |    1.274 |  1.272 | 106
M-08 |  1.565 |  1.494 |    1.494 |    1.477 |  1.487 | 127
M-09 |  4.375 |  4.400 |    1.497 |    4.388 |  4.395 | 127
M-10 |  5.300 |  5.307 |    1.497 |    5.312 |  5.306 | 127
参数存储估算：FP32=84 bytes，INT8权重+FP32偏置=36 bytes，压缩=2.33x


## 结果解读

量化误差不是单纯由位宽决定：错误校准让异常输入大量饱和，误差远高于代表性校准。QAT 的每一步都真实经过 fake-quant 前向和 STE backward，因此可以微调码点附近的参数。INT8 权重矩阵输出说明这里确实执行了舍入和截断，而非只计算一个压缩比例。

## 生产边界

本例是 per-tensor 对称量化和 fake integer 推理，没有零点、per-channel 权重、bias accumulator scale、算子融合或真实 INT8 kernel。生产部署要按目标 CPU/NPU 支持选择量化方案，在留出集校准并逐层观察饱和率，还要实际 benchmark 端到端延迟；小模型可能因量化/反量化开销反而更慢。

## 最小回归测试

In [6]:
assert len(records) >= 5  # 保证量化案例包含足够多传感器样本
assert bad_saturation > good_saturation  # 保证不具代表性校准真实造成更多输入饱和
assert bad_ptq_mse > good_ptq_mse  # 保证代表性校准修正同模型量化误差
assert int(good_codes["weight1_int"].min()) >= -127 and int(good_codes["weight1_int"].max()) <= 127  # 保证手写权重整数码落在对称 INT8 范围
assert qat_trace[-1][1] < qat_trace[0][1]  # 保证 QAT 的量化路径损失经过真实训练下降
assert all(torch.isfinite(parameter).all() for parameter in qat_model.parameters())  # 保证 STE 参数更新保持有限
assert int8_bytes < fp_bytes  # 保证权重 INT8 后参数存储估算确实下降